# Tool Use Deep Dive

**Live online course — instructor walkthrough notebook**

This notebook follows the course lecture notes on tool use. We build a small reminder assistant from scratch: Claude can ask the system for the current time, do precise date math, and set a reminder. By the end we layer in batching, structured data extraction, streaming, and the two built-in tools (text editor + web search).

Each section has:
- **Lecture notes** (markdown) — what to explain on screen.
- **Demo code** — cells to run live so students see real output.
- **🏫 During class** callouts — specific instructor actions, talking points, and live variations.

---

## Agenda

1. Introducing Tool Use
2. Project Overview — the reminder assistant
3. Tool Functions
4. Tool Schemas
5. Handling Message Blocks (multi-block responses)
6. Sending Tool Results
7. Multi-Turn Conversations with Tools
8. Implementing Multiple Turns (the run_conversation loop)
9. Using Multiple Tools
10. The Batch Tool
11. Tools for Structured Data
12. Fine-Grained Tool Calling (streaming)
13. The Text Editor Tool (built-in)
14. The Web Search Tool (built-in)
15. Recap + practice exercises

## 0. Setup (do this before class starts)

1. Install dependencies:
   ```bash
   pip install anthropic python-dotenv
   ```
2. Create a file named `.env` in the same directory as this notebook containing:
   ```
   ANTHROPIC_API_KEY="sk-ant-...your-key..."
   ```
3. Add `.env` to `.gitignore` so it is never committed to version control.

> **🏫 During class:** Before running the first cell, open `.env` and show students what it looks like (blur the key). Emphasize: **never paste the API key directly into a notebook cell** — `.env` + `python-dotenv` keeps secrets out of source control.

In [ ]:
# Install packages (uncomment if not already installed)
# %pip install anthropic python-dotenv

The next cell wires up the SDK and runs a 3-line sanity check before any API call:

1. `load_dotenv()` reads `.env` and copies `ANTHROPIC_API_KEY` into the process environment. The key never appears in the notebook.
2. `client = Anthropic()` constructs the SDK client. With no arguments it picks up `ANTHROPIC_API_KEY` from the environment automatically.
3. `model = "claude-sonnet-4-6"` — a single named constant we reference everywhere. Swap this one line and every demo switches model.
4. The three `print(...)` lines confirm: SDK installed, model name pinned, `.env` was found. If `Key loaded: False` prints, fix `.env` before running anything else.

In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic
import anthropic
import os

load_dotenv()

client = Anthropic()

# Default workhorse for the course. Use claude-haiku-4-5 for cheap/fast iteration
# or claude-opus-4-7 for the hardest reasoning tasks.
model = "claude-sonnet-4-6"

print("SDK version:", anthropic.__version__)
print("Model:", model)
print("Key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

We start with the simple text-only `chat()` helper from the intro notebook. In **§5** we will deliberately refactor it to return the full `Message` object — that is the moment students learn about multi-block responses.

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """Send messages to Claude and return the assistant text."""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        # Sonnet 4.6 / Opus 4.7 default to adaptive extended thinking. Disable
        # it so every demo here returns a clean, predictable content list —
        # important once we start inspecting tool_use blocks.
        "thinking": {"type": "disabled"},
    }
    if system is not None:
        params["system"] = system
    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text

---
# 1. Introducing Tool Use

**Tool use** is how Claude reaches *outside* its training data — the model decides it needs a fact it doesn't have, asks the host application to fetch it, then writes the final answer using what came back.

### The default limitation
Claude only knows what it saw during training. It has **no live information**: no current date/time, no fresh data, no access to your databases or APIs. Without help, anything it writes about "now" is an educated guess.

### The 5-step flow

```
1. User → Server: "What's the weather in Toronto?"
2. Server → Claude: prompt + list of tools the model can request
3. Claude  : decides it needs live data → emits a tool_use block
             { name: "get_weather", input: { city: "Toronto" } }
4. Server  : runs get_weather("Toronto") → calls the real weather API
5. Server → Claude: tool_result block with the API response
6. Claude → Server → User: a natural-language answer using that data
```

### Key concept
Claude does not call your function. Claude **asks** for it. Your code runs the function and feeds the result back. Tools turn Claude into the orchestrator of an external data lookup.

### Demo: ask Claude for live information — with no tools attached

We deliberately give Claude no tools. The point is to *show the gap*: ask for the current time and watch the model say it cannot. This is the motivation for everything that follows.

In [ ]:
messages = []
add_user_message(messages, "What is the current date and time, right now? Answer in one sentence.")
print(chat(messages))

> **🏫 During class:**
> 1. Run the cell. Highlight the hedge in the answer ("I don't have access to real-time information").
> 2. Say out loud: *"This isn't a model bug — it's the API doing the right thing. The model has no clock; we have to give it one."*
> 3. Ask the room: *"What's another piece of information Claude can't know without help?"* Collect 2–3 examples (today's stock price, a database row, the user's calendar) — these are all candidate **tools**.

---
# 2. Project Overview — the reminder assistant

**Goal:** Teach Claude to set time-based reminders.

**Target interaction:**
> User: *"Set a reminder for my doctor's appointment, a week from Thursday."*  
> Claude: *"Done — I'll remind you on April 30th at 9am."*

### Three problems Claude can't solve alone

| Problem | Why tools help |
|---|---|
| **No clock** | Claude knows the current *date* of training, not the *exact moment* of the request. |
| **Date math is unreliable** | Ask for *"379 days from January 13th, 1973"* and Claude often slips by a day. |
| **No reminder mechanism** | Claude can describe a reminder but can't actually schedule one. |

### Three tools we will build

| Tool | Job |
|---|---|
| `get_current_datetime` | Returns the system's current date + time. |
| `add_duration_to_datetime` | Adds an offset (days/weeks/months) to a datetime. |
| `set_reminder` | Schedules the reminder (mocked — just prints). |

Implementation approach: **one tool at a time**, then orchestrate them together.

### Demo: same target prompt, no tools

We send the exact prompt the assistant should eventually handle — with no tools attached. Watch which of the three failures shows up: a vague answer, a wrong date, an apology, or all three. This makes the missing capabilities concrete before we write any tool code.

In [ ]:
messages = []
add_user_message(
    messages,
    "Set a reminder for my doctor's appointment. It's 177 days after January 1, 2050.",
)
print(chat(messages))

> **🏫 During class:**
> 1. Run the cell and read the response out loud.
> 2. Point at the response and say: *"Two failures in one answer — it can't actually schedule, and it might be off by a day on the date."* Verify the date in a calendar app live (Jan 1, 2050 + 177 days = June 27, 2050).
> 3. Ask: *"Which of the three problems would each of our planned tools fix?"* Map back to the table above.

---
# 3. Tool Functions

**Tool functions** are plain Python functions that Claude can ask the host to run.

### Required characteristics
- **Descriptive names** for the function and every argument. Claude reads the names and uses them to decide *whether* to call the tool and *how* to fill in arguments.
- **Input validation** with `raise ValueError(...)` on bad input. Error messages are forwarded back to Claude as tool results, so write them like documentation — tell Claude what went wrong and what to try instead.
- **Return a string or a JSON-serializable object.** The host serializes the return value before sending it back.

### Workflow
```
Claude needs data → calls tool fn → success: result goes back
                                  → raises:  error message goes back → Claude retries with corrected args
```

Treat tool functions like **any production utility**: validate inputs, fail loudly, document the shape of the output.

### Demo: define and call the first two tool functions — directly, no Claude

Before we wire these into Claude, we run them like any normal Python function. This proves the contract works in isolation. Notice how `unit` validates and raises on a bad value — that error message is exactly what Claude will see if it picks the wrong unit, and it's specific enough to self-correct.

In [ ]:
from datetime import datetime, timedelta

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)
    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    else:
        raise ValueError(
            f"Unsupported time unit: {unit!r}. Use one of: seconds, minutes, hours, days, weeks."
        )
    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")
    return f"Reminder set for {timestamp}"

# Direct calls — no Claude involved yet
print("Now :", get_current_datetime("%Y-%m-%d %H:%M"))
print("+177:", add_duration_to_datetime("2050-01-01", duration=177, unit="days"))

# Trigger validation — Claude would see this string in a tool_result and retry.
try:
    add_duration_to_datetime("2050-01-01", duration=10, unit="fortnights")
except ValueError as e:
    print("Validation error:", e)

> **🏫 During class:**
> 1. Run the cell. Point at the validation error and say: *"Claude reads this exact string. So we wrote it like a hint, not a stack trace."*
> 2. Ask: *"Why does `add_duration_to_datetime` take an `input_format` arg?"* (Answer: lets Claude work with whatever date string the user typed without us doing string surgery.)
> 3. Variation: change the error message to a stack-trace-style `"KeyError: fortnights"` and discuss how that would degrade Claude's recovery.

---
# 4. Tool Schemas

A **tool schema** is a JSON description of the tool function. It's what Claude actually sees — your Python source code is *not* sent. The schema is how Claude knows the tool exists, when to use it, and how to fill in the arguments.

### Three required fields

| Field | What it is | How Claude uses it |
|---|---|---|
| `name` | Function identifier (must match Python). | Routing on the server side. |
| `description` | 3–4 sentences: what it does, **when to use it**, what it returns. | Decides *whether* to call the tool. |
| `input_schema` | JSON Schema for the arguments. | Decides *how* to fill arguments. |

### JSON Schema in 60 seconds
JSON Schema is a generic data-validation spec (predates LLMs). Tool calling adopted it because it's already standard, supports nested objects, lists, enums, defaults, and required fields. For each property describe `type`, `description`, optional `default`.

### Schema generation trick
Don't hand-write schemas. Paste your Python function into Claude.ai with: *"Write a valid JSON tool schema for this function, following the Anthropic tool use docs (attached)."* Then drop the result here.

### Implementation conventions
- Name the schema `[function_name]_schema`.
- Wrap with `ToolParam(...)` from `anthropic.types` so static type checkers catch shape mistakes.

### Demo: write the schemas for our three tools, then print one

We write all three schemas now so the rest of the notebook can reuse them. Note how the descriptions are **paragraphs, not labels** — they tell Claude *when* the tool is appropriate. Compare the verbosity of `add_duration_to_datetime_schema` to a generic docstring; the extra detail is what produces correct argument selection.

In [ ]:
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": (
        "Returns the current system date and time formatted according to the specified format string. "
        "Use this whenever you need to know what 'now' is — for example, before computing a future date or "
        "timestamping a record. The default format returns 'YYYY-MM-DD HH:MM:SS'."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "Python strftime format string. Examples: '%Y-%m-%d' for date only, "
                    "'%H:%M:%S' for time only, '%B %d, %Y' for 'April 26, 2026'. "
                    "Defaults to '%Y-%m-%d %H:%M:%S'."
                ),
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
})

add_duration_to_datetime_schema = ToolParam({
    "name": "add_duration_to_datetime",
    "description": (
        "Adds a duration to a datetime string and returns the resulting datetime. "
        "Use this for any future-date or past-date arithmetic — e.g., '20 days from today', "
        "'3 weeks before this meeting'. Output format is 'Weekday, Month DD, YYYY HH:MM:SS AM/PM'."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The starting datetime string, formatted per input_format.",
            },
            "duration": {
                "type": "number",
                "description": "Amount to add. Positive = future, negative = past. Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "One of: 'seconds', 'minutes', 'hours', 'days', 'weeks'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "strptime format for parsing datetime_str. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
})

set_reminder_schema = ToolParam({
    "name": "set_reminder",
    "description": (
        "Schedules a reminder notification for the user at a specific timestamp. "
        "Use this once you have BOTH the reminder content and a concrete timestamp — "
        "never call it with a placeholder timestamp. The reminder persists across app restarts."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The reminder text the user will see (e.g., 'Doctor's appointment').",
            },
            "timestamp": {
                "type": "string",
                "description": "ISO 8601 timestamp YYYY-MM-DDTHH:MM:SS for when to fire the reminder.",
            },
        },
        "required": ["content", "timestamp"],
    },
})

print("Schema preview:")
import json as _json
print(_json.dumps(get_current_datetime_schema, indent=2))

> **🏫 During class:**
> 1. Run the cell. Point at the printed schema and walk through `name`, `description`, `input_schema.properties` one by one.
> 2. Read the `set_reminder` description out loud, especially *"never call it with a placeholder timestamp"* — this is how you constrain Claude's behavior **without** writing code.
> 3. Variation: shorten `add_duration_to_datetime`'s description to one sentence and predict what Claude will get wrong (often: picks the wrong `unit`).

---
# 5. Handling Message Blocks

Now we make our first request **with tools attached**. Two things change.

### Change 1 — add the `tools=` argument
Pass the schema list. Claude only sees tools you explicitly include.

```python
client.messages.create(
    model=model,
    messages=[...],
    tools=[get_current_datetime_schema],
)
```

### Change 2 — responses are now multi-block
Up to now `response.content` was a list with one `TextBlock`. With tools enabled it can also contain **`ToolUseBlock`** entries:

```python
response.content = [
    TextBlock(text="Let me check the time."),
    ToolUseBlock(id="toolu_...", name="get_current_datetime", input={"date_format": "%H:%M"}),
]
```

### Why our helpers must change
The history we send back to Claude must include the **whole assistant message** — every block, not just the text. If we drop the tool_use block, Claude can't pair the upcoming tool_result with the request that triggered it. So:

- `chat()` now returns the full `Message` object (not just text).
- `add_assistant_message()` accepts either a string or a `Message` and stores its `.content` list.
- We add a `text_from_message()` helper for when we just want the human-readable text.

**Critical:** the API stores nothing between requests. Conversation history with tool calls is *your* job, exactly as before — just with richer content blocks.

### Demo: refactor the helpers, then call Claude with one tool

We redefine `chat`, `add_user_message`, `add_assistant_message`, and add `text_from_message`. The first request after the refactor asks for the current time — Claude returns a `tool_use` block instead of a final answer. We print the raw `content` list so students see the new shape.

In [ ]:
from anthropic.types import Message

def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })
    return messages

def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None,
         tools=None, tool_choice=None):
    """Return the full Message object so callers can inspect content blocks."""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "thinking": {"type": "disabled"},
    }
    if system is not None:
        params["system"] = system
    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences
    if tools is not None:
        params["tools"] = tools
    if tool_choice is not None:
        params["tool_choice"] = tool_choice
    return client.messages.create(**params)

def text_from_message(message):
    return "\n".join(b.text for b in message.content if b.type == "text")

# First tool-enabled request
messages = []
add_user_message(messages, "What is the current time in HH:MM format?")
response = chat(messages, tools=[get_current_datetime_schema])

print("stop_reason:", response.stop_reason)
print("content blocks:")
for block in response.content:
    print(" -", block.type, "|", block)

> **🏫 During class:**
> 1. Run the cell. Point at `stop_reason: 'tool_use'` — *"this is how the SDK signals 'I want a tool, not a final answer'."*
> 2. Read out the `tool_use` block fields: `id`, `name`, `input`. Note that `input` is a dict matching the schema we wrote.
> 3. Ask: *"Could the same response also contain a text block?"* (Yes — Claude often narrates before requesting the tool. We saved both via `.content`.)
> 4. Quick takeaway: *"The model didn't run the function. It asked us to."*

---
# 6. Sending Tool Results

Once Claude has emitted a `tool_use` block, the host runs the function and sends the result back in a follow-up request.

### The tool_result block
```python
{
    "type": "tool_result",
    "tool_use_id": "toolu_...",   # MUST match the id from the tool_use block
    "content": json.dumps(output), # tool output as a string
    "is_error": False,             # True if the function raised
}
```

### `tool_use_id` matters
If Claude emits **two** tool_use blocks in one message (e.g., "check the time AND check the date format"), each tool_result must reference the right id. The id is how the model pairs a request with its result.

### Where the block goes
Tool results travel as **user** messages. The conversation now looks like:

```
user      → "What time is it?"
assistant → [text + tool_use]
user      → [tool_result]            ← we send this
assistant → "It's 14:32."
```

And **every** request includes the *full* history plus the original `tools=` list — the API is stateless.

### Demo: hand-execute the tool, send the result, get the final answer

We continue from the previous cell's `response`. We extract the tool_use block, call the actual Python function, build a tool_result block, append it as a user message, and call Claude again. The final response should be a plain text block with the time — no more tool_use.

In [ ]:
import json

# Save the assistant message with its tool_use block
add_assistant_message(messages, response)

# Extract the tool_use block
tool_use = next(b for b in response.content if b.type == "tool_use")
print("Claude requested:", tool_use.name, tool_use.input)

# Execute the actual Python function
tool_output = get_current_datetime(**tool_use.input)
print("Tool returned :", tool_output)

# Build the tool_result block (note: id must match)
tool_result_block = {
    "type": "tool_result",
    "tool_use_id": tool_use.id,
    "content": json.dumps(tool_output),
    "is_error": False,
}

# Send the follow-up. tool_result rides INSIDE a user message's content list.
add_user_message(messages, [tool_result_block])

# Tools must still be present — the API is stateless
final = chat(messages, tools=[get_current_datetime_schema])
add_assistant_message(messages, final)

print("\nFinal answer:\n", text_from_message(final))
print("\nstop_reason:", final.stop_reason)

> **🏫 During class:**
> 1. Run the cell. Walk the audience through each step in the printout: *request → execute → result → final answer*.
> 2. Open `messages` in a new cell (`messages`) and show the four entries: user, assistant (text+tool_use), user (tool_result), assistant (final). Say: *"This is what gets sent on every future call. We carry it forever."*
> 3. Variation to try live: change `tool_use.id` to a wrong string before building the result block. The API rejects the request — reinforces why the id is mandatory.

---
# 7. Multi-Turn Conversations with Tools

Real prompts often need **chains** of tool calls. The user asks one question; Claude needs two, three, or more tool round-trips before it has enough information to answer.

### Example chain
> User: *"What day will it be 103 days from today?"*

1. Claude requests `get_current_datetime` → host runs it → returns today's date.
2. Claude requests `add_duration_to_datetime(today, 103, days)` → host runs it → returns target date.
3. Claude writes the answer.

Two tool calls, three Claude requests. The host can't predict in advance how many round-trips a prompt needs — the **prompt itself decides**.

### The pattern
```
while True:
    response = chat(messages, tools=[...])
    add_assistant_message(messages, response)
    if response.stop_reason != "tool_use":
        break
    tool_results = run_tools(response)
    add_user_message(messages, tool_results)
```

Two helpers we still need:
- `run_tool(name, input)` — dispatches to the right Python function.
- `run_tools(message)` — walks every `tool_use` block in the assistant message and produces one `tool_result` per call.

### Demo: ask a question that *requires* two tool calls and watch the chain

We don't have the loop yet, but we can already prove the chain is needed. We'll send a multi-step prompt with both schemas attached and inspect the first response: it will request a tool, **not** answer the question. That single round-trip is enough motivation for the loop in the next section.

In [ ]:
messages = []
add_user_message(messages, "What full date will it be 103 days from today?")

response = chat(
    messages,
    tools=[get_current_datetime_schema, add_duration_to_datetime_schema],
)

print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "text":
        print("text:", block.text)
    elif block.type == "tool_use":
        print("tool_use:", block.name, block.input)

> **🏫 During class:**
> 1. Run the cell. Point out: *"It asked for `get_current_datetime` first — the natural first step. It hasn't done the math yet."*
> 2. Ask: *"What would happen if we stopped here and showed the user this response?"* (They'd see no answer, just a tool request.)
> 3. Set up the next section: *"We need a loop that keeps going until `stop_reason` is `'end_turn'`. That's what we build next."*

---
# 8. Implementing Multiple Turns

Now the loop. Three helpers:

| Helper | Job |
|---|---|
| `run_tool(name, input)` | Dispatch on tool name; call the matching Python function. |
| `run_tools(message)` | For every `tool_use` block in the assistant message, run the tool, wrap the output (or error) as a `tool_result`. |
| `run_conversation(messages)` | The `while True:` loop — calls Claude, runs requested tools, feeds results back, breaks when `stop_reason != 'tool_use'`. |

### Stop reasons
`stop_reason` tells you *why* generation halted. The values you'll see in tool flows:

| Value | Meaning |
|---|---|
| `tool_use` | Claude wants you to run a tool and call back. |
| `end_turn` | Claude is done; the message is the final answer. |
| `max_tokens` | Hit the cap — you may need a larger `max_tokens`. |
| `stop_sequence` | A configured stop sequence fired. |

We branch on `tool_use` vs. anything-else.

### Error handling
Wrap each tool call in `try/except`. If the function raises, send the error string back as the tool result with `is_error=True`. Claude will see the error and can recover — retry with corrected args, or apologize and ask the user.

### Demo: implement the three helpers and run the chain

After this cell runs, the same prompt that previously stopped at one tool request now resolves all the way to a final natural-language answer. The loop printed `>>> tool: get_current_datetime` then `>>> tool: add_duration_to_datetime`, then the final text — watch for those two tool lines in order.

In [ ]:
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")

def run_tools(message):
    tool_requests = [b for b in message.content if b.type == "tool_use"]
    results = []
    for req in tool_requests:
        print(f"  >>> tool: {req.name}({req.input})")
        try:
            output = run_tool(req.name, req.input)
            results.append({
                "type": "tool_result",
                "tool_use_id": req.id,
                "content": json.dumps(output),
                "is_error": False,
            })
        except Exception as e:
            results.append({
                "type": "tool_result",
                "tool_use_id": req.id,
                "content": f"Error: {e}",
                "is_error": True,
            })
    return results

def run_conversation(messages, tools):
    while True:
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)
        text = text_from_message(response)
        if text:
            print(text)
        if response.stop_reason != "tool_use":
            return messages
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

# Drive the chain
messages = []
add_user_message(messages, "What full date will it be 103 days from today?")
run_conversation(messages, tools=[
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
])

> **🏫 During class:**
> 1. Run the cell. Trace the printed lines aloud: *"tool 1 → tool 2 → final text. The loop ran 3 Claude calls and 2 Python functions before exiting."*
> 2. Ask: *"What if we removed the `if response.stop_reason != 'tool_use': return` line?"* (Infinite loop — we'd send tool_results in a row.)
> 3. Variation: change the prompt to *"What time is it AND what's the date 50 days from now AND 100 days from now?"* — watch the loop run more turns.

---
# 9. Using Multiple Tools

Once `run_conversation` exists, **adding a new tool is mechanical**:

1. Add the schema to the `tools=[...]` list at the call site.
2. Add a branch in `run_tool` mapping the name to the Python function.
3. Implement (or stub) the function.

We already wrote the third tool's schema (`set_reminder_schema`) and added it to `run_tool`. So this section is mostly: **wire it in and watch the model coordinate three tools to satisfy the original target prompt.**

### Why this scales
Claude doesn't know our tool framework. It only knows: *here is a list of capabilities, here is the user's request*. Going from 1 tool to 10 tools is purely a question of writing better schemas (so Claude picks correctly) — the orchestration code stays the same.

### Demo: drive the original target prompt to completion

We send the prompt from §2 — the one Claude couldn't handle without tools — with all three schemas attached. Watch the chain: the model resolves the date math first, then calls `set_reminder` with a concrete ISO timestamp. The `set_reminder` mock prints a confirmation, and the final text summarizes what was scheduled.

In [ ]:
messages = []
add_user_message(
    messages,
    "Set a reminder for my doctor's appointment. It's 177 days after January 1, 2050.",
)

run_conversation(messages, tools=[
    get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
])

> **🏫 During class:**
> 1. Run the cell. Compare the output to §2's failed attempt — *same prompt, totally different outcome.*
> 2. Point at the `set_reminder` print line: *"That's a real side effect. In production it's an HTTP call to a notification service."*
> 3. Ask: *"Why did Claude not call `get_current_datetime` here?"* (Because the user's prompt anchored on Jan 1, 2050 — no need for `now()`. The schema descriptions guided Claude to skip it.)
> 4. Variation: change the prompt to *"Set a reminder for the dentist 3 weeks from today"* and re-run — now Claude *does* need `get_current_datetime`.

---
# 10. The Batch Tool

Claude *can* emit multiple `tool_use` blocks in a single assistant message — enabling **parallel** tool execution. In practice it rarely chooses to. With sequentially-runnable tools, this means an unnecessary round-trip per call.

### Trick: a `batch_tool` wrapper
Expose one extra tool whose argument is a **list of (tool name, args)** pairs. Now the most natural way for Claude to express "do these three things" is one call to `batch_tool`. Server-side we unpack the list and run each invocation.

### Schema sketch
```json
{
  "name": "batch_tool",
  "description": "Invoke multiple other tool calls simultaneously",
  "input_schema": {
    "type": "object",
    "properties": {
      "invocations": {
        "type": "array",
        "items": {
          "type": "object",
          "properties": {
            "name": {"type": "string"},
            "arguments": {"type": "string", "description": "JSON-encoded args"}
          },
          "required": ["name", "arguments"]
        }
      }
    },
    "required": ["invocations"]
  }
}
```

### Result
One assistant message → one tool_result → done. What used to be N round-trips becomes one.

### Demo: send a prompt that asks for several independent date calculations at once

We add `batch_tool_schema` and a `run_batch` dispatcher to `run_tool`. Then we ask Claude to compute three offset dates from a single anchor. Without batch, Claude would do three sequential round-trips. With batch, it should make **one** `batch_tool` call containing three invocations.

In [ ]:
batch_tool_schema = ToolParam({
    "name": "batch_tool",
    "description": (
        "Invoke multiple other tool calls simultaneously when they are independent. "
        "Use this whenever the user asks for several pieces of information that can be "
        "computed in parallel — it produces all results in a single round-trip."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "List of independent tool calls to execute.",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string", "description": "Tool name to invoke."},
                        "arguments": {"type": "string", "description": "JSON-encoded args."},
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
})

def run_batch(invocations):
    outputs = []
    for inv in invocations:
        outputs.append({
            "name": inv["name"],
            "output": run_tool(inv["name"], json.loads(inv["arguments"])),
        })
    return outputs

# Re-bind run_tool to know about batch
_prev_run_tool = run_tool
def run_tool(tool_name, tool_input):
    if tool_name == "batch_tool":
        return run_batch(tool_input["invocations"])
    return _prev_run_tool(tool_name, tool_input)

messages = []
add_user_message(
    messages,
    "Starting from January 1, 2050, what dates are 30 days, 90 days, and 365 days later?",
)

run_conversation(messages, tools=[
    add_duration_to_datetime_schema,
    batch_tool_schema,
])

> **🏫 During class:**
> 1. Run the cell. Count the printed `>>> tool:` lines: ideally **one** — a single `batch_tool` call with three invocations inside.
> 2. Open `messages` and find the assistant's tool_use block. Its `input` should be the `invocations` list with all three sub-calls.
> 3. Variation: comment out `batch_tool_schema` from the `tools=` list and re-run — watch it become 3 sequential round-trips.
> 4. Talking point: *"This isn't a feature of Claude. It's a higher-level abstraction we built. The schema is the API contract."*

---
# 11. Tools for Structured Data

Tool calling is also a clean way to extract **structured JSON** from unstructured input. Compared to the prompt-based pre-fill + stop-sequence trick from the intro notebook, this approach is:

| | Pre-fill + stop sequence | Tool extraction |
|---|---|---|
| Reliability | Good | **Better** — schema-validated |
| Setup cost | Low | Higher (write the schema) |
| Best for | Quick scripts, experiments | Production pipelines |

### Pattern
1. Define a tool whose **input_schema** is the data shape you want.
2. Force Claude to call it with `tool_choice={"type": "tool", "name": "..."}`.
3. Read `response.content[0].input` — that's your structured dict, validated.

Notice: there is **no** tool execution. We never ship `tool_result`s. We use the schema as a typed-output contract.

### Demo: extract a structured ticket from free-form text

We give Claude a short bug report and force it to call `extract_ticket`. The result is a dict matching the schema — ready to insert into a ticketing system. We never run an `extract_ticket` Python function; the tool exists purely as a typed contract.

In [ ]:
extract_ticket_schema = ToolParam({
    "name": "extract_ticket",
    "description": "Extract a structured support ticket from free-form text.",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_name": {"type": "string"},
            "priority": {
                "type": "string",
                "enum": ["low", "medium", "high", "urgent"],
            },
            "category": {
                "type": "string",
                "enum": ["billing", "bug", "feature_request", "other"],
            },
            "summary": {
                "type": "string",
                "description": "One-sentence summary of the issue.",
            },
        },
        "required": ["customer_name", "priority", "category", "summary"],
    },
})

raw_email = """
Hi support — this is Priya Shah. Our checkout page has been throwing a 500 error
for the past hour, and customers are bouncing. We're losing real money. Please
look at this ASAP.
"""

messages = [{"role": "user", "content": raw_email}]
response = chat(
    messages,
    tools=[extract_ticket_schema],
    tool_choice={"type": "tool", "name": "extract_ticket"},
)

tool_use = next(b for b in response.content if b.type == "tool_use")
ticket = tool_use.input
print("Structured ticket:")
print(json.dumps(ticket, indent=2))

> **🏫 During class:**
> 1. Run the cell. Show that `ticket` is a dict, ready to `INSERT INTO tickets ...`.
> 2. Remove `tool_choice=...` and re-run. Sometimes Claude returns a text block answering the email instead of calling the tool. *"`tool_choice` is what makes this reliable."*
> 3. Ask: *"What if the input email had no priority cue?"* (Claude picks based on the schema's `enum` and the email's tone. Test it live by stripping "ASAP".)

---
# 12. Fine-Grained Tool Calling (streaming)

When you stream a response, tool arguments arrive as **deltas** too — character by character.

### Stream events with tools
| Event | Meaning |
|---|---|
| `content_block_start` (tool_use) | A new tool_use block begins. |
| `input_json` / `input_json_delta` | Partial JSON for the tool's `input` field. Each event has both `partial_json` (the new chunk) and a cumulative snapshot. |
| `content_block_stop` | Block finished. |

### Default behavior — validated streaming
By default, the API **buffers** JSON chunks until each top-level key is complete and validates against the schema before sending. You see bursts of deltas instead of a smooth stream. Invalid JSON Claude generates (e.g. `"undefined"` instead of `null`) gets coerced to a string — the structure stays valid.

### Fine-grained mode — raw streaming
Opt in with the beta header `fine-grained-tool-streaming-2025-05-14`. The API stops validating and forwards every delta as Claude produces it. You get classic typewriter streaming for tool args. The trade: invalid JSON now lands on your client — you must handle parse errors yourself.

| | Default | Fine-grained |
|---|---|---|
| Latency to first chunk | Slower (buffers + validates) | Immediate |
| JSON validity | Guaranteed | **Your responsibility** |
| Use case | Most production code | Live tool-arg UI, early dispatch |

### Demo: stream a tool call and watch the input_json deltas

We use `client.messages.stream(...)` directly (not our `chat()` helper) and route different event types to different print prefixes. You'll see `[text]` for narration, then `[tool_use:name]` when the block starts, then a series of `[delta]` lines as the JSON arguments stream in. The smoothness of the deltas (or lack of it) is the difference between fine-grained and default.

In [ ]:
messages = [{
    "role": "user",
    "content": "What date is exactly 60 days after April 1, 2030?",
}]

with client.messages.stream(
    model=model,
    max_tokens=400,
    messages=messages,
    tools=[add_duration_to_datetime_schema],
    thinking={"type": "disabled"},
) as stream:
    for event in stream:
        if event.type == "text":
            print("[text]", event.text, flush=True)
        elif event.type == "content_block_start":
            block = event.content_block
            if block.type == "tool_use":
                print(f"[tool_use:{block.name}] block start", flush=True)
        elif event.type == "input_json" and event.partial_json:
            print("[delta]", repr(event.partial_json), flush=True)
        elif event.type == "content_block_stop":
            print("[block_stop]", flush=True)

    final = stream.get_final_message()

print("\nFinal stop_reason:", final.stop_reason)
print("Final blocks    :", [b.type for b in final.content])

> **🏫 During class:**
> 1. Run the cell. Trace the order: `text` (sometimes), `tool_use start`, several `delta` lines, `block_stop`.
> 2. Point at the `[delta]` lines and say: *"Notice they arrive in bursts — that's the API validating each top-level key before flushing."*
> 3. Variation: add `extra_headers={"anthropic-beta": "fine-grained-tool-streaming-2025-05-14"}` to the stream call to flip on fine-grained streaming — the deltas should start arriving smoother and earlier.
> 4. Takeaway: *"Fine-grained = perceived speed at the cost of validity guarantees. Default is the right call until you're rendering tool args live."*

---
# 13. The Text Editor Tool (built-in)

The **text editor tool** is one of the few tools whose schema is **built into Claude**. You don't write the JSON — you send a 2-field stub and the API expands it server-side into the full schema with all the file operations.

### What it does
Lets Claude operate on files like a code editor: `view`, `str_replace`, `create`, `insert`, `undo_edit`. With this one tool plus a competent file-system implementation on your side, Claude can act as an autonomous coding agent.

### Stub schema
```python
{
    "type": "text_editor_20250728",
    "name": "str_replace_based_edit_tool",
}
```
The `type` value is **versioned per Claude model family** — newer models support newer commands. Always pull the right type string from the docs for your target model.

### What you still have to write
Claude only emits requests. You implement the actual file system: `view(path)`, `str_replace(path, old, new)`, `create(path, content)`, etc., and route by the `command` field of the tool input. The course's `005_text_editor_tool.ipynb` has a complete `TextEditorTool` class you can drop in as a reference.

### When it's worth it
Out-of-the-box, you get something close to a custom AI code editor without building a UI — great for batch refactors, codemod pipelines, automated docs updates.

### Demo: send the stub schema and inspect what Claude requests

We don't wire up a file-system implementation here — the goal is to show that the **stub** is enough for Claude to start emitting structured editor commands. We ask it to fix a typo in a (fictional) file. It will return a `tool_use` block with `command="view"` first (it wants to read before editing), proving the schema-stub-expansion mechanism works.

In [ ]:
text_editor_schema = {
    "type": "text_editor_20250728",
    "name": "str_replace_based_edit_tool",
}

messages = [{
    "role": "user",
    "content": "There's a typo in /workspace/notes.txt. The word 'recieve' should be 'receive'. Please fix it.",
}]

response = client.messages.create(
    model=model,
    max_tokens=500,
    messages=messages,
    tools=[text_editor_schema],
    thinking={"type": "disabled"},
)

print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "text":
        print("\n[text]", block.text)
    elif block.type == "tool_use":
        print("\n[tool_use]", block.name)
        print("  input:", json.dumps(block.input, indent=2))

> **🏫 During class:**
> 1. Run the cell. Point at the `command` field in the tool_use input — likely `"view"`. *"It wants to read the file first, even though we told it the typo. Good engineering instinct, learned from the docs."*
> 2. Ask: *"What command would we expect on the next round-trip if we returned the file's contents?"* (`str_replace`.)
> 3. Reference: open `005_text_editor_tool.ipynb` in another tab and skim the `TextEditorTool` class — walk through `_validate_path` and `_backup_file` for safety.
> 4. Takeaway: *"Built-in tools = Claude knows the schema. Custom tools = you write the schema. Both flow through the same `tool_use`/`tool_result` shape."*

---
# 14. The Web Search Tool (built-in)

The **web search tool** is the second built-in. Unlike the text editor, you don't even implement the function — the Anthropic backend runs the search itself. You only declare the schema.

### Schema
```python
{
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],   # optional whitelist
}
```

| Field | Why it matters |
|---|---|
| `max_uses` | Caps total searches per request. Prevents runaway costs. |
| `allowed_domains` | Restricts results to a trusted whitelist. Huge for medical, legal, technical use cases. |

### What comes back
The `response.content` list is richer than usual:

| Block type | Holds |
|---|---|
| `text` | Claude's narrative answer. |
| `server_tool_use` | The query string Claude sent. |
| `web_search_tool_result` | List of pages found (title + URL + raw snippet). |
| `text` with `citations` | Specific sentences linked back to source URLs. |

### UI rendering pattern
Render the text. Show the search-result list as a sidebar. When a citation appears, mark the sentence and link to its source. This is the pattern used by Claude.ai, Perplexity, etc.

### Quality trick
Whitelisting domains is the cheapest possible quality lever. *"Best exercise for leg muscle?"* on the open web is content-marketing soup. Same prompt restricted to `nih.gov` returns peer-reviewed material.

### Demo: medical question restricted to nih.gov, then inspect the block types

We ask a fitness question with `allowed_domains=["nih.gov"]`. Claude makes one or more searches, then synthesizes an answer with citations. We print the block types so students see how rich the content list becomes — not just text, but searches, results, and citation metadata.

In [ ]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 3,
    "allowed_domains": ["nih.gov"],
}

messages = [{
    "role": "user",
    "content": "What's the most effective resistance exercise for building quadriceps muscle? One short paragraph.",
}]

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[web_search_schema],
    thinking={"type": "disabled"},
)

print("stop_reason:", response.stop_reason)
print("\nBlock types in response:")
for i, block in enumerate(response.content):
    print(f"  {i}. {block.type}")

print("\n--- Narrative text ---")
for block in response.content:
    if block.type == "text":
        print(block.text)
        if getattr(block, "citations", None):
            for cite in block.citations:
                print(f"   → cite: {getattr(cite, 'url', '?')}")

> **🏫 During class:**
> 1. Run the cell. Read the printed block types out loud — students see `server_tool_use`, `web_search_tool_result`, and `text` (often more than one).
> 2. Point at the citation URLs: *"Every cited sentence links back to the source page. That's how you build a 'show your work' UI."*
> 3. Variation: remove `allowed_domains` and re-run. The answer style typically shifts — broader sources, more hedging. *"That whitelist isn't censorship; it's a quality filter."*
> 4. Tie it together: *"Built-in tools (text editor, web search) and custom tools live in the same content-block format. Once you've handled `tool_use` + `tool_result` once, you've handled them all."*

---
# 15. Recap + practice exercises

### Recap (run through these out loud)
- **Tools = orchestrated lookups.** Claude decides it needs data; the host fetches; Claude composes the answer.
- **Three pieces per custom tool:** Python function (with validation), JSON schema (with a *paragraph* of description), dispatch in `run_tool`.
- **Multi-block messages:** assistant content can be `text` + `tool_use[]`. History must store the full content list.
- **`tool_use_id` pairs requests with results.** The API rejects mismatches.
- **`run_conversation` loop:** call → if `stop_reason=='tool_use'` execute and append results → repeat. Otherwise return.
- **Adding tools is mechanical:** schema in the `tools=` list, branch in `run_tool`, function. No re-architecting.
- **`batch_tool`** turns sequential calls into one round-trip when invocations are independent.
- **Structured data via tools:** schema = your typed contract; force with `tool_choice`; read `tool_use.input` directly.
- **Streaming tool args:** `input_json` events give you partial JSON. Default validates; fine-grained beta forwards every delta.
- **Built-ins:** text editor (you implement) + web search (Anthropic implements). Same content-block protocol as custom tools.

### Exercises (do the first in class, assign the rest)
1. **Add a fourth tool:** `list_reminders()` that returns the reminders you've set so far (use a global list). Wire it in and ask Claude *"What reminders do I have?"*.
2. **Schema audit:** rewrite `set_reminder_schema`'s description to be **one sentence** and re-run §9's demo. Note which arguments Claude gets wrong.
3. **Structured extraction:** build an `extract_invoice` tool whose schema includes `vendor`, `total`, `line_items[]` (each with `description` + `amount`). Force the call on a 3-line invoice email.
4. **Streaming UI:** stream the §9 reminder demo and print each `tool_use` request as soon as it arrives. Compare perceived latency to the non-streaming version.
5. **Web search agent:** build a small "medical question answerer" that uses `web_search` restricted to `nih.gov` and `cdc.gov`, with a system prompt that forces citations into every sentence.

In [ ]:
# Exercise 1 scaffold — finish this live in class together.
#
# REMINDERS = []
#
# def set_reminder(content, timestamp):
#     REMINDERS.append({"content": content, "timestamp": timestamp})
#     return f"Reminder set for {timestamp}"
#
# def list_reminders():
#     return REMINDERS
#
# list_reminders_schema = ToolParam({
#     "name": "list_reminders",
#     "description": "Returns all reminders the user has scheduled in this session.",
#     "input_schema": {"type": "object", "properties": {}, "required": []},
# })
#
# # 1. Add a branch in run_tool for "list_reminders".
# # 2. Add list_reminders_schema to the run_conversation tools list.
# # 3. Ask: "What reminders do I have so far?"